# Chapter 17 — Signal vs Noise: SNR & Filtering
*Track 4: Engineers*

## 🎯 Learning Objectives
- Quantify signal-to-noise ratio and its probabilistic interpretation
- Implement moving average and exponential smoothing filters
- Apply Kalman filter concepts for state estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.signal as signal

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Signal and Noise

A measurement is:
$$y[t] = s[t] + n[t]$$

where $s[t]$ is the true signal and $n[t] \sim N(0, \sigma_n^2)$ is noise.

**Signal-to-Noise Ratio (SNR):**
$$\text{SNR} = \frac{P_{signal}}{P_{noise}} = \frac{\sigma_s^2}{\sigma_n^2}$$

$$\text{SNR}_{dB} = 10\log_{10}\left(\frac{P_{signal}}{P_{noise}}\right)$$

In [ ]:
# Simulate noisy signal
t = np.linspace(0, 10, 500)
signal_true = np.sin(2*np.pi*0.5*t) + 0.3*np.sin(2*np.pi*2.0*t)
noise_levels = [0.1, 0.5, 1.5]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sigma_n in zip(axes, noise_levels):
    noise = rng.normal(0, sigma_n, len(t))
    y = signal_true + noise
    snr_linear = signal_true.var() / noise.var()
    snr_db = 10 * np.log10(snr_linear)
    ax.plot(t, y, alpha=0.7, lw=0.8, label="Noisy")
    ax.plot(t, signal_true, "r-", lw=2, label="True signal")
    ax.set_title(f"σ_noise={sigma_n}
SNR={snr_db:.1f} dB")
    ax.legend(fontsize=8)
plt.suptitle("Effect of Noise on Signal Quality", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Moving Average as Low-Pass Filter

An order-$M$ moving average:
$$\hat s[t] = \frac{1}{M}\sum_{k=0}^{M-1} y[t-k]$$

The variance of the MA estimate:
$$\text{Var}(\hat s[t]) = \frac{\sigma_n^2}{M}$$

→ SNR improves by factor $M$.

In [ ]:
sigma_n = 0.8
noise = rng.normal(0, sigma_n, len(t))
y_noisy = signal_true + noise

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, M in zip(axes.flat, [1, 5, 20, 50]):
    if M == 1:
        y_filt = y_noisy
    else:
        kernel = np.ones(M) / M
        y_filt = np.convolve(y_noisy, kernel, mode="same")
    residual_var = np.var(y_filt - signal_true)
    snr_improvement = sigma_n**2 / residual_var
    ax.plot(t, y_noisy, alpha=0.4, lw=0.8, label="Noisy")
    ax.plot(t, signal_true, "g-", lw=1.5, label="True")
    ax.plot(t, y_filt, "r-", lw=2, label=f"MA(M={M})")
    ax.set_title(f"M={M}, SNR gain={snr_improvement:.1f}x")
    ax.legend(fontsize=8)
plt.suptitle("Moving Average Filtering — Variance Reduction", fontweight="bold")
plt.tight_layout(); plt.show()

## 3–6. Exponential Smoothing, Kalman Filter, Practice

In [ ]:
# Exponential smoothing (EMA)
def ema(y, alpha):
    s = np.zeros_like(y)
    s[0] = y[0]
    for i in range(1, len(y)):
        s[i] = alpha * y[i] + (1-alpha) * s[i-1]
    return s

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, alpha in zip(axes, [0.1, 0.3, 0.8]):
    y_smooth = ema(y_noisy, alpha)
    ax.plot(t, y_noisy, alpha=0.4, lw=0.8)
    ax.plot(t, signal_true, "g-", lw=1.5, label="True")
    ax.plot(t, y_smooth, "r-", lw=2, label=f"EMA α={alpha}")
    ax.set_title(f"α={alpha}")
    ax.legend(fontsize=9)
plt.suptitle("Exponential Moving Average Smoothing")
plt.tight_layout(); plt.show()

In [ ]:
# Kalman filter: 1D constant velocity model
# State: [position, velocity], Measurement: position only
dt = t[1] - t[0]
F = np.array([[1, dt], [0, 1]])  # state transition
H = np.array([[1, 0]])           # measurement matrix
Q = np.diag([1e-4, 1e-4])       # process noise covariance
R_kf = np.array([[sigma_n**2]]) # measurement noise covariance

x_est = np.zeros((len(t), 2))
P_est = np.eye(2) * 1.0
x_est[0] = [y_noisy[0], 0]

for k in range(1, len(t)):
    # Predict
    x_pred = F @ x_est[k-1]
    P_pred = F @ P_est @ F.T + Q
    # Update
    S = H @ P_pred @ H.T + R_kf
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_est[k] = x_pred + K @ (y_noisy[k:k+1] - H @ x_pred)
    P_est = (np.eye(2) - K @ H) @ P_pred

plt.figure(figsize=(10, 4))
plt.plot(t, y_noisy, alpha=0.4, lw=0.8, label="Noisy")
plt.plot(t, signal_true, "g-", lw=1.5, label="True signal")
plt.plot(t, x_est[:, 0], "r-", lw=2, label="Kalman filter")
plt.title("Kalman Filter — 1D State Estimation")
plt.legend(); plt.tight_layout(); plt.show()

kf_mse = np.mean((x_est[:, 0] - signal_true)**2)
noisy_mse = np.mean((y_noisy - signal_true)**2)
print(f"Noisy MSE: {noisy_mse:.4f}  →  Kalman MSE: {kf_mse:.4f}")